# Track labels in multi-channel time-lapse stacks

For each time-lapse (`ch0 = brightfield`, `ch1..chN = binary label masks`) under `stack_outputs`:

1. For each label channel (`MASK_CHANNELS`), extract the mask and fill holes, then label objects per frame.
2. Link objects over time independently per label channel with TrackMate's Advanced Kalman tracker (no splitting/merging).
3. Save a multi-channel output (`ch0 = brightfield`, each subsequent channel = that input channel's linked track labels, written back to the same channel index) as a 32-bit ImageJ TIFF with the pixel calibration embedded. Tracked objects keep a consistent id across frames; objects that were never linked to a track are given the value **-1**.
4. Write one morphology CSV per label channel (`morphology_channel{N}.csv`): every object id, its source image, the mask channel it came from, and morphological parameters (area in µm² and px², eccentricity, solidity, ...) at each timepoint.


In [1]:
import logging
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
import imagej
from imagej import Mode
import scyjava as sj
from scipy import ndimage as ndi
from skimage.measure import label as cc_label, regionprops_table

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOGGER = logging.getLogger("tracking")


In [ ]:
# --- Configuration -----------------------------------------------------------
INPUT_ROOT = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\NXD11\combined_stacks")
OUTPUT_ROOT = INPUT_ROOT.parent / "stack_outputs_tracked"

# Channel roles inside each input stack (T, C, Y, X).
BRIGHTFIELD_CHANNEL = 0
MASK_CHANNELS = [1, 2]  # channel indices holding label masks; each is tracked independently

# Pixel calibration. Value given as cm; everything else is derived from it.
PIXEL_SIZE_CM = 0.0064500
PIXEL_SIZE_UM = PIXEL_SIZE_CM * 10_000.0  # 1 cm = 10,000 µm
PIXEL_AREA_UM2 = PIXEL_SIZE_UM ** 2

# TrackMate Advanced Kalman tracker settings.
INITIAL_SEARCH_RADIUS = 200.0   # frame-to-frame linking distance (LINKING_MAX_DISTANCE)
SEARCH_RADIUS = 400.0          # Kalman search radius (KALMAN_SEARCH_RADIUS)
MAX_FRAME_GAP = 2
QUALITY_PENALTY = 0.2          # feature penalty applied to QUALITY
ALLOW_TRACK_SPLITTING = False
ALLOW_TRACK_MERGING = False

print("Output root:", OUTPUT_ROOT)
print(f"Pixel size: {PIXEL_SIZE_CM} cm = {PIXEL_SIZE_UM} µm/px  ->  {PIXEL_AREA_UM2:.4f} µm²/px")


Output root: Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\stack_outputs_tracked
Pixel size: 0.00645 cm = 64.5 µm/px  ->  4160.2500 µm²/px


In [3]:
# --- Mask preparation and morphology -----------------------------------------

def prepare_instance_labels(mask_stack):
    """Fill holes in the binary mask and label objects per frame.

    Parameters
    ----------
    mask_stack : np.ndarray
        (T, Y, X) mask channel. Any non-zero pixel is treated as foreground.

    Returns
    -------
    np.ndarray
        (T, Y, X) uint16 stack where each object in each frame has a unique
        per-frame integer label (background = 0).
    """
    num_frames = mask_stack.shape[0]
    labelled = np.zeros(mask_stack.shape, dtype=np.uint16)
    for frame_index in range(num_frames):
        foreground = mask_stack[frame_index] > 0
        foreground = ndi.binary_fill_holes(foreground)
        labelled[frame_index] = cc_label(foreground).astype(np.uint16)
    return labelled


def measure_morphology(linked_labels, source_name, pixel_size_um, mask_channel):
    """Compute per-object, per-timepoint morphology from a tracked label stack.

    Parameters
    ----------
    linked_labels : np.ndarray
        (T, Y, X) stack where every object carries the same id across frames.
        Unlinked objects (value -1) are ignored.
    source_name : str
        Name of the source image (stored in the ``image`` column).
    pixel_size_um : float
        Physical pixel size in micrometres.
    mask_channel : int
        Input channel index these objects were segmented/tracked from (stored
        in the ``mask_channel`` column).

    Returns
    -------
    pandas.DataFrame
        One row per (object_id, timepoint) with morphological parameters.
    """
    properties = ("label", "area", "perimeter", "eccentricity", "solidity", "extent",
                  "major_axis_length", "minor_axis_length", "orientation",
                  "equivalent_diameter_area", "centroid")
    per_frame_tables = []
    for frame_index in range(linked_labels.shape[0]):
        frame_labels = np.clip(linked_labels[frame_index], 0, None)  # drop -1 unlinked objects
        if frame_labels.max() == 0:
            continue
        table = pd.DataFrame(regionprops_table(frame_labels, properties=properties))
        table.insert(0, "timepoint", frame_index)
        per_frame_tables.append(table)

    if not per_frame_tables:
        return pd.DataFrame()

    morphology = pd.concat(per_frame_tables, ignore_index=True)
    morphology = morphology.rename(columns={
        "label": "object_id",
        "area": "area_pixels",
        "perimeter": "perimeter_pixels",
        "major_axis_length": "major_axis_length_pixels",
        "minor_axis_length": "minor_axis_length_pixels",
        "equivalent_diameter_area": "equivalent_diameter_pixels",
        "centroid-0": "centroid_y",
        "centroid-1": "centroid_x",
    })
    morphology["area_um2"] = morphology["area_pixels"] * (pixel_size_um ** 2)
    morphology["perimeter_um"] = morphology["perimeter_pixels"] * pixel_size_um
    morphology["major_axis_length_um"] = morphology["major_axis_length_pixels"] * pixel_size_um
    morphology["minor_axis_length_um"] = morphology["minor_axis_length_pixels"] * pixel_size_um
    morphology["equivalent_diameter_um"] = morphology["equivalent_diameter_pixels"] * pixel_size_um
    morphology["aspect_ratio"] = morphology["major_axis_length_pixels"] / morphology["minor_axis_length_pixels"].replace(0, np.nan)
    morphology.insert(0, "image", source_name)
    morphology.insert(1, "mask_channel", mask_channel)

    ordered = ["object_id", "image", "mask_channel", "timepoint",
               "area_pixels", "area_um2",
               "perimeter_pixels", "perimeter_um",
               "eccentricity", "solidity", "extent", "aspect_ratio",
               "major_axis_length_pixels", "major_axis_length_um",
               "minor_axis_length_pixels", "minor_axis_length_um",
               "equivalent_diameter_pixels", "equivalent_diameter_um",
               "orientation", "centroid_y", "centroid_x"]
    return morphology[ordered].sort_values(["object_id", "timepoint"]).reset_index(drop=True)

In [4]:
# --- TrackMate linking -------------------------------------------------------

def run_trackmate_linking(label_stack_path):
    """Link labelled objects over time with TrackMate's Advanced Kalman tracker.

    Parameters
    ----------
    label_stack_path : str or Path
        Path to a single-channel (T, Y, X) instance-label TIFF.

    Returns
    -------
    dict
        Keys: ``tracks_df`` (per-spot dataframe), ``model``, ``settings``,
        ``image_plus`` (the TrackMate ImagePlus, for XML export).
    """
    IJ = sj.jimport("ij.IJ")
    HashMap = sj.jimport("java.util.HashMap")
    Integer = sj.jimport("java.lang.Integer")
    Double = sj.jimport("java.lang.Double")
    Model = sj.jimport("fiji.plugin.trackmate.Model")
    Settings = sj.jimport("fiji.plugin.trackmate.Settings")
    TrackMate = sj.jimport("fiji.plugin.trackmate.TrackMate")
    Logger = sj.jimport("fiji.plugin.trackmate.Logger")
    LabelImageDetectorFactory = sj.jimport("fiji.plugin.trackmate.detection.LabelImageDetectorFactory")
    AdvancedKalmanTrackerFactory = sj.jimport("fiji.plugin.trackmate.tracking.kalman.AdvancedKalmanTrackerFactory")

    image_plus = IJ.openImage(str(label_stack_path))
    if image_plus is None:
        raise RuntimeError(f"Could not open label stack: {label_stack_path}")
    num_timepoints = int(image_plus.getNFrames())
    if num_timepoints <= 1:
        num_timepoints = int(image_plus.getStackSize())
    image_plus.setDimensions(1, 1, num_timepoints)
    image_plus.setOpenAsHyperStack(True)

    model = Model()
    model.setLogger(Logger.VOID_LOGGER)
    settings = Settings(image_plus)
    settings.detectorFactory = LabelImageDetectorFactory()
    detector_settings = HashMap()
    detector_settings.put("TARGET_CHANNEL", Integer.valueOf(1))
    detector_settings.put("SIMPLIFY_CONTOURS", False)
    settings.detectorSettings = detector_settings

    settings.trackerFactory = AdvancedKalmanTrackerFactory()
    tracker_settings = HashMap(settings.trackerFactory.getDefaultSettings())
    tracker_settings.put("KALMAN_SEARCH_RADIUS", Double.valueOf(float(SEARCH_RADIUS)))
    tracker_settings.put("LINKING_MAX_DISTANCE", Double.valueOf(float(INITIAL_SEARCH_RADIUS)))
    tracker_settings.put("MAX_FRAME_GAP", Integer.valueOf(int(MAX_FRAME_GAP)))
    tracker_settings.put("ALLOW_TRACK_SPLITTING", bool(ALLOW_TRACK_SPLITTING))
    tracker_settings.put("ALLOW_TRACK_MERGING", bool(ALLOW_TRACK_MERGING))
    quality_penalty = HashMap()
    quality_penalty.put("QUALITY", Double.valueOf(float(QUALITY_PENALTY)))
    tracker_settings.put("LINKING_FEATURE_PENALTIES", quality_penalty)
    tracker_settings.put("GAP_CLOSING_FEATURE_PENALTIES", quality_penalty)
    settings.trackerSettings = tracker_settings
    settings.addAllAnalyzers()

    trackmate = TrackMate(model, settings)
    if not trackmate.checkInput():
        raise RuntimeError(f"TrackMate input rejected: {trackmate.getErrorMessage()}")
    if not trackmate.process():
        raise RuntimeError(f"TrackMate processing failed: {trackmate.getErrorMessage()}")

    track_model = model.getTrackModel()
    spot_records = []
    for track_id in track_model.trackIDs(True):
        for spot in track_model.trackSpots(track_id):
            spot_records.append({
                "track_id": int(track_id),
                "t": int(float(spot.getFeature("FRAME"))),
                "y": float(spot.getFeature("POSITION_Y")),
                "x": float(spot.getFeature("POSITION_X")),
                "quality": float(spot.getFeature("QUALITY")),
            })
    tracks_df = pd.DataFrame(spot_records, columns=["track_id", "t", "y", "x", "quality"])
    return {"tracks_df": tracks_df, "model": model, "settings": settings, "image_plus": image_plus}


def paint_linked_labels(instance_labels, tracks_df):
    """Re-paint per-frame instance labels with globally consistent track ids.

    Each tracked object is looked up by its spot centroid and every pixel of
    that object in that frame is set to ``track_id + 1`` (label 0 = background).
    Foreground objects that were never linked to a track are set to -1.
    """
    linked = np.zeros(instance_labels.shape, dtype=np.int32)
    num_frames, num_y, num_x = instance_labels.shape
    for track_id, frame, y, x in tracks_df[["track_id", "t", "y", "x"]].to_numpy():
        frame_index = int(frame)
        if not 0 <= frame_index < num_frames:
            continue
        y_index = int(np.clip(round(float(y)), 0, num_y - 1))
        x_index = int(np.clip(round(float(x)), 0, num_x - 1))
        source_label = int(instance_labels[frame_index, y_index, x_index])
        if source_label > 0:
            linked[frame_index, instance_labels[frame_index] == source_label] = int(track_id) + 1
    linked[(instance_labels > 0) & (linked == 0)] = -1
    return linked


In [5]:
# --- Per-image processing ----------------------------------------------------

def save_tracked_imagej_tiff(output_path, brightfield, linked_labels_by_channel):
    """Save a multi-channel ImageJ TIFF with calibration.

    Channel 0 is brightfield; each subsequent channel holds the tracked labels
    for the corresponding entry of ``linked_labels_by_channel`` (same order as
    ``MASK_CHANNELS``, so tracked labels land back in their original channel).
    Written as 32-bit float so unlinked objects can carry the value -1.
    """
    channel_arrays = [brightfield.astype(np.float32)] + [labels.astype(np.float32) for labels in linked_labels_by_channel]
    output_stack = np.stack(channel_arrays, axis=1)  # (T, C, Y, X)
    resolution = (1.0 / PIXEL_SIZE_CM, 1.0 / PIXEL_SIZE_CM)  # pixels per cm
    tifffile.imwrite(
        str(output_path),
        output_stack,
        imagej=True,
        resolution=resolution,
        metadata={"axes": "TCYX", "unit": "cm", "mode": "composite"},
    )


def process_single_image(image_path, output_dir):
    """Process one multi-channel stack: track each label channel independently,
    save outputs, return a dict of ``{mask_channel: morphology_df}``."""
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = image_path.stem

    stack = tifffile.imread(str(image_path))  # (T, C, Y, X)
    brightfield = stack[:, BRIGHTFIELD_CHANNEL]

    linked_labels_by_channel = []
    morphology_by_channel = {}
    for mask_channel in MASK_CHANNELS:
        mask = stack[:, mask_channel]
        instance_labels = prepare_instance_labels(mask)

        temp_handle, temp_name = tempfile.mkstemp(suffix=".tif")
        os.close(temp_handle)
        try:
            tifffile.imwrite(temp_name, instance_labels)
            result = run_trackmate_linking(temp_name)
        finally:
            try:
                os.remove(temp_name)
            except OSError:
                pass

        linked_labels = paint_linked_labels(instance_labels, result["tracks_df"])
        linked_labels_by_channel.append(linked_labels)
        morphology_by_channel[mask_channel] = measure_morphology(linked_labels, stem, PIXEL_SIZE_UM, mask_channel)

        n_tracks = int(result["tracks_df"]["track_id"].nunique())
        LOGGER.info("%s: channel %d -> %d tracks", stem, mask_channel, n_tracks)

    output_image_path = output_dir / f"{stem}_tracked.tif"
    save_tracked_imagej_tiff(output_image_path, brightfield, linked_labels_by_channel)
    LOGGER.info("%s: saved -> %s", stem, output_image_path.name)
    return morphology_by_channel

In [6]:
# --- Initialise ImageJ / TrackMate (run once) --------------------------------
ij = imagej.init("sc.fiji:fiji", mode=Mode.HEADLESS, add_legacy=True)
print("ImageJ version:", ij.getVersion())


ImageJ version: 2.17.0/1.54p


In [7]:

image_paths = sorted(INPUT_ROOT.glob("*.tif"))
morphology_frames_by_channel = {channel: [] for channel in MASK_CHANNELS}

for image_path in image_paths:
    try:
        print(f"Processing image {image_path.name}")
        morphology_by_channel = process_single_image(image_path, OUTPUT_ROOT)
        for channel, morphology in morphology_by_channel.items():
            if not morphology.empty:
                morphology_frames_by_channel[channel].append(morphology)
    except Exception as error:
        LOGGER.error("Failed on %s: %s", image_path.name, error)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for channel, frames in morphology_frames_by_channel.items():
    if frames:
        channel_morphology = pd.concat(frames, ignore_index=True)
        csv_path = OUTPUT_ROOT / f"morphology_channel{channel}.csv"
        channel_morphology.to_csv(csv_path, index=False)
        LOGGER.info("Wrote %d rows -> %s", len(channel_morphology), csv_path)
    else:
        LOGGER.warning("No morphology rows produced for channel %d.", channel)


print("Done.")

Processing image A1_Neurons_NXD8_labels.tif


2026-10-06 18:13:41,163 INFO A1_Neurons_NXD8_labels: channel 1 -> 155 tracks
2026-10-06 18:42:30,154 INFO A1_Neurons_NXD8_labels: channel 2 -> 481 tracks
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\fluorofate_environment\Lib\site-packages\tifffile\tifffile.py:3711: UserWarning: <tifffile.TiffWriter 'A1_Neurons_NXD8_…els_tracked.tif'> truncating ImageJ file
  self._write_remaining_pages()
2026-10-06 18:48:19,291 INFO A1_Neurons_NXD8_labels: saved -> A1_Neurons_NXD8_labels_tracked.tif


Processing image A2_NCO013T_NXD8_labels.tif


2026-10-06 18:57:34,180 INFO A2_NCO013T_NXD8_labels: channel 1 -> 109 tracks
2026-10-06 19:38:48,764 INFO A2_NCO013T_NXD8_labels: channel 2 -> 616 tracks
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\fluorofate_environment\Lib\site-packages\tifffile\tifffile.py:3711: UserWarning: <tifffile.TiffWriter 'A2_NCO013T_NXD8_…els_tracked.tif'> truncating ImageJ file
  self._write_remaining_pages()
2026-10-06 19:44:38,905 INFO A2_NCO013T_NXD8_labels: saved -> A2_NCO013T_NXD8_labels_tracked.tif


Processing image A3_NCO015T_NXD8_labels.tif


...

Processing image D5_mLCNEC23_NXD8_labels.tif


2026-10-07 17:13:38,136 INFO D5_mLCNEC23_NXD8_labels: channel 1 -> 270 tracks
2026-10-07 17:37:21,333 INFO D5_mLCNEC23_NXD8_labels: channel 2 -> 428 tracks
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\fluorofate_environment\Lib\site-packages\tifffile\tifffile.py:3711: UserWarning: <tifffile.TiffWriter 'D5_mLCNEC23_NXD…bels_tracked.tif'> truncating ImageJ file
  self._write_remaining_pages()
2026-10-07 17:42:53,980 INFO D5_mLCNEC23_NXD8_labels: saved -> D5_mLCNEC23_NXD8_labels_tracked.tif


Processing image D6_mLCNEC26_NXD8_labels.tif


2026-10-07 18:36:56,633 INFO D6_mLCNEC26_NXD8_labels: channel 1 -> 833 tracks
2026-10-07 18:48:44,487 INFO D6_mLCNEC26_NXD8_labels: channel 2 -> 255 tracks
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\fluorofate_environment\Lib\site-packages\tifffile\tifffile.py:3711: UserWarning: <tifffile.TiffWriter 'D6_mLCNEC26_NXD…bels_tracked.tif'> truncating ImageJ file
  self._write_remaining_pages()
2026-10-07 18:54:14,308 INFO D6_mLCNEC26_NXD8_labels: saved -> D6_mLCNEC26_NXD8_labels_tracked.tif
2026-10-07 18:54:19,731 INFO Wrote 53645 rows -> Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\stack_outputs_tracked\morphology_channel1.csv
2026-10-07 18:54:23,925 INFO Wrote 98777 rows -> Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\stack_outputs_tracked\morphology_channel2.csv


Done.
